In [1]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
sys.path.append(str(project_root))

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from torchvision import transforms
import pytorch_lightning as pl
from torch.utils.data import Dataset, Subset, DataLoader

from constants import *
from extra_functions import set_seed
from data_setup import ImageDataset, HierImageDataset
from hierarchy import Hierarchy

/Users/alexandermichaeltjhin/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/alexandermichaeltjhin/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from collections import Counter
class HierImageDataset(Dataset):
    
    """
    A custom PyTorch Dataset for creating a dataset with hierarchical labels

    This class handles:
    - Optional image transforms for data augmentation

    Args:
        base_dataset (ImageDataset): The original dataset from which the hierarchical dataset will be built. 
        groups (list): A list of string lists, where each inner list defines how the final nodes are grouped at a given
                       coarser level. The order of the list is important: the first corresponds to the finest
                       grouping, and the last to the broadest. 
        coarse_names (list): A list of string lists, where each inner list defines the names of the groups at a given
                            coarser level. The order of the list is important: the fist corresponds to the finest grouping
                            and the last to the broadest. 
        image_transforms (callable, optional): Image transformations (e.g., data augmentations) to apply. Defaults to None.

        
    Attributes:
        data_directory (str): Path to the dataset root directory (from base_dataset).
        data_subdirectories (list of str): Subdirectories with additional images (from base_dataset).
        seed (int): Random seed used for sampling (from base_dataset).
        class_names (list): List by level with sorted class names included in the dataset.
        class_sizes (torch.Tensor): List by level with the actual sampled size per class.
        class_ids (list): List by level with numeric ID for each class (aligned with `class_names`).
        image_paths (list): List of file paths to all sampled images (from base_dataset).
        labels (list): List by level with numeric class IDs corresponding to each image.
        image_resolution (int): Size to which each image is resized (from base_dataset).
        image_transforms (callable or None): Image transformations applied during training or inference.
        format_file (str): Format of images files. Default: '.tif (from base_dataset)' 
        
    """
       
    def __init__(self, base_dataset,
                #  , groups,coarse_names,
                 adjacency_graph,levels,
                 image_transforms = None,
                 leaves_only=False
        ):
        
        self.data_directory = base_dataset.data_directory
        self.data_subdirectories = base_dataset.data_subdirectories
        self.seed = base_dataset.seed
        self.image_resolution =  base_dataset.image_resolution
        self.image_paths = base_dataset.image_paths
        self.label_to_ids = {name: i for i, name in enumerate(hier_adjacency_graph.keys())}
        self.levels = levels
        
        self.labels = [self.label_to_ids[label] for label in base_dataset.labels]
        self.class_names = []
        self.class_sizes = []
        self.class_ids = []

        # Build hierarchy object
        self.hierarchy = self._build_hierarchy(adjacency_graph, self.label_to_ids)
        # Other class initializations
        self.image_resolution = base_dataset.image_resolution
        self.image_transforms = image_transforms
        if leaves_only:
            self._filter_leaves()
        
    def __len__(self):

        """
        Returns the number of samples in the Dataset.
        """

        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert('L')
        if self.image_transforms:
            image = self.image_transforms(image)

        image = image.repeat(3, 1, 1)

        label_node = self.labels[idx]

        # path from root to labeled node
        path = self.hierarchy.get_path_to_root(label_node)

        # supervision per node
        targets = {}
        masks = {}

        for parent in path[:-1]:
            children = self.hierarchy.children(parent)

            target_child = path[path.index(parent) + 1]

            targets[parent] = children.index(target_child)

            masks[parent] = [1] * len(children)

        return {
            "image": image,
            "label_node": label_node,
            "path": path,
            "targets": targets,
            "masks": masks
        }
    
    def _build_hierarchy(self, adjacency_graph, label_to_ids):
        """
        Constructs a Hierarchy object from an adjacency graph.
        
        Args:
            adjacency_graph (dict): Dictionary mapping parent -> list of children
            
        Returns:
            Hierarchy: Populated hierarchy object
        """
        hierarchy = Hierarchy()
        
        all_children = set()
        for _, kids in adjacency_graph.items():
            all_children.update(kids)

        # root = node never appearing as child
        root = [n for n in adjacency_graph if n not in all_children][0]

        hierarchy.add_node(label_to_ids[root], root)

        def dfs(parent):
            for child in adjacency_graph[parent]:
                hierarchy.add_node(label_to_ids[child], child, label_to_ids[parent])
                dfs(child)

        dfs(root)
        
        return hierarchy
    

    def _filter_leaves(self):
        """
        Filters image_paths and labels to only include samples
        whose label corresponds to a leaf node in the hierarchy.
        """
        leaf_ids = set(
            node_id for node_id in self.hierarchy.nodes
            if self.hierarchy.is_leaf(node_id)
        )

        filtered_paths, filtered_labels = zip(
            *[(path, label) for path, label in zip(self.image_paths, self.labels)
            if label in leaf_ids]
        ) if self.image_paths else ([], [])

        removed = len(self.image_paths) - len(filtered_paths)

        self.image_paths = list(filtered_paths)
        self.labels = list(filtered_labels)

        print(f"[leaves_only] Kept {len(self.image_paths)} samples | Removed {removed} non-leaf samples")

    def collate_fn(self, batch):
        image = torch.stack([b["image"] for b in batch])
        label_node = [F.one_hot(torch.tensor(b["label_node"], dtype=torch.long), num_classes=len(self.hierarchy)).float() for b in batch]
        label_node = torch.stack(label_node)
        # label_node = [b["label_node"] for b in batch]
        path = [b["path"] for b in batch]
        targets = [b["targets"] for b in batch]
        masks = [b["masks"] for b in batch]
        return {
            "image": image,
            "label_node": label_node,
            "path": path,
            "targets": targets,
            "masks": masks
        }

    def print_dataset_details(self):
        """
        Prints the class distribution of the Dataset.
        """
        print(f'\nTotal Dataset: Size = {len(self)} | Levels = {self.levels}')

        # Get all unique labels (leaf nodes) from the dataset
        leaf_counts = dict(Counter(self.labels))
        
        # Calculate counts for all nodes (including parent nodes)
        all_node_counts = {}
        
        # For each node in the hierarchy, calculate its count
        for node_id in self.hierarchy.nodes.keys():
            if self.hierarchy.is_leaf(node_id):
                all_node_counts[node_id] = leaf_counts.get(node_id, 0)
            else:
                # Parent nodes: sum of all descendant leaf counts
                leaf_descendants = self.hierarchy.descendants(node_id) + [node_id]
                count = 0
                for leaf_id in leaf_descendants:
                    count += leaf_counts.get(leaf_id, 0)
                all_node_counts[node_id] = count

        print(f"all_node_counts: {all_node_counts}\n")

        # Group nodes by their depth in hierarchy
        nodes_by_level = {}
        for node_id in self.hierarchy.nodes.keys():
            depth = self.hierarchy.nodes[node_id].depth
            if depth not in nodes_by_level:
                nodes_by_level[depth] = []
            nodes_by_level[depth].append(node_id)

        print(f"nodes_by_level: {nodes_by_level}\n")

        # Print statistics for each level
        for level in range(self.levels + 1):
            if level not in nodes_by_level:
                continue
            
            level_nodes = sorted(nodes_by_level[level], key=lambda x: self.hierarchy.nodes[x].node_id)
            level_counts = {node_id: all_node_counts[node_id] for node_id in level_nodes}
            
            print(f"\n------------------------Level {level}------------------------")
            
            for node_id in level_nodes:
                node = self.hierarchy.nodes[node_id]
                count = all_node_counts[node_id]
                
                # Skip nodes with zero count
                if count == 0:
                    continue
                
                class_prop = count / len(self)
                node_type = "Leaf" if self.hierarchy.is_leaf(node_id) else "Parent"
                
                print(
                    f'Level: {level} | Class Name: {node.name:20s} | Class Label: {node_id:3d} | '
                    f'Type: {node_type:6s} | Count: {count:6d} | Prop: {class_prop:.2f}'
                )
            
            # Check for missing labels (only relevant for leaf nodes at this level)
            leaf_nodes_at_level = [n for n in level_nodes if self.hierarchy.is_leaf(n)]
            if leaf_nodes_at_level:
                missing_count = sum(1 for label in self.labels 
                                if label is None and self.hierarchy.depth(label) == level)
                if missing_count > 0:
                    print(f'\nImportant: At level {level} there are {missing_count} samples with missing labels\n')
                    
    
    def split_train_test_val(
        self, train_prop: float = 0.7, val_prop: float = 0.1, test_prop: float = 0.2
    ):

        """
        Returns indices corresponding to the train, validation and test subsets of the Dataset.

        Args:
            trian_prop (float): Proportion of samples to allocate to the train subset.
            val_prop (float): Proportion of samples to allocate to the validation subset.
            test_prop (float): Proportion of samples to allocate to the test subset.
        """

        train_split, val_split, test_split = random_split(
            range(len(self)),
            lengths = [train_prop, val_prop, test_prop],
            generator = torch.Generator().manual_seed(self.seed)
        )

        return train_split.indices, val_split.indices, test_split.indices
    
    def append_image_transforms(
        self, image_transforms: transforms.Compose = None, replace: bool = False
        ):        
        """
        Appends image transformations to existing transformation pipeline or replaces.
        If multiple `ToTensor()` transformations are included in the resulting pipeline, only the last instance is kept.
        If there are no `ToTensor()` transformations in the resulting pipeline, it is appended.

        Args:
            image_transforms(transfors.Compose, optional): Iterable of image transformations to append.
            replace (bool): Specifies whether to replace with or append the above image_transforms.
        """
        
        if image_transforms is None:
            if self.image_transforms is None:
                image_transforms_list = []
            else: 
                image_transforms_list = self.image_transforms.transforms
        else:
            if replace:
                image_transforms_list = image_transforms.transforms
            else:
                image_transforms_list = self.image_transforms.transforms + image_transforms.transforms

        image_transforms_cleaned = []
        to_tensor_indices = [i for i, tf in enumerate(image_transforms_list) if isinstance(tf, transforms.ToTensor)]

        if to_tensor_indices:
            last_idx = to_tensor_indices[-1]
            image_transforms_cleaned = [tf for i, tf in enumerate(image_transforms_list) if not isinstance(tf, transforms.ToTensor) or i == last_idx]
        else:
            image_transforms_cleaned = image_transforms_list + [transforms.ToTensor()]

        self.image_transforms = transforms.Compose(image_transforms_cleaned)
                    
    def create_dataloaders(
        self, batch_size: int, train_indices, val_indices, test_indices,
        image_transforms: transforms.Compose = None, transform_val: bool = False, 
        train_sample_weights: torch.tensor = None
    ):
        
        """
        Creates the train, validatinon and test DataLoaders required for training a PyTorch model.
        If `train_sample_weights` is specified, they are supplied to WeightedRandomSampler for the train subset.

        Args:
            batch_size (int): Sizes of batches to process samples in DataLoader.
            train_indices (list): Indices corresponding to the train subset of the Dataset.
            val_indices (list): Indices corresponding to the validation subset of the Dataset.
            test_indices (list): Indices corresponding to the test subset of the Dataset.
            image_transforms (transforms.Compose, optional): Additional image transformations for the train subset.
            transform_val (bool): Specifies whether to apply train image transformations to the validation subset.
            train_sample_weights (torch.tensor, optional): Contains weights for each sample in the train subset.
        """

        if image_transforms is not None:
            dataset_aug = copy.deepcopy(self)
            dataset_aug.append_image_transforms(
                image_transforms = image_transforms, verbose = False
            )
            train_dataset = Subset(dataset_aug, train_indices)
            if transform_val:
                val_dataset = Subset(dataset_aug, val_indices)
            else:
                val_dataset = Subset(self, val_indices)
        else:
            train_dataset = Subset(self, train_indices)
            val_dataset = Subset(self, val_indices)
        
        test_dataset = Subset(self, test_indices)

        if train_sample_weights is None:
            train_loader = DataLoader(train_dataset, batch_size = batch_size, shuffle = True, 
                collate_fn=self.collate_fn,generator = torch.Generator().manual_seed(self.seed)
            )
        else:
            train_loader = DataLoader(
                train_dataset, batch_size = batch_size, collate_fn=self.collate_fn,
                sampler = WeightedRandomSampler(train_sample_weights, num_samples = len(train_sample_weights), replacement = True)
            )

        val_loader = DataLoader(
            val_dataset, batch_size = batch_size, sampler = SequentialSampler(val_dataset),
            collate_fn=self.collate_fn
        )
        test_loader = DataLoader(
            test_dataset, batch_size = batch_size, sampler = SequentialSampler(test_dataset),
            collate_fn=self.collate_fn
        )

        return train_loader, val_loader, test_loader   
        


In [4]:
class Node:
    def __init__(self, node_id, name, parent=None):
        self.node_id = node_id
        self.name = name
        self.parent = parent
        self.children = []
        self.depth = None

    def add_child(self, child_id):
        """Add a child node ID to this node's children list."""
        if child_id not in self.children:
            self.children.append(child_id)
    
    def __repr__(self):
        return f"Node(id={self.node_id}, name={self.name}, parent={self.parent}, children={self.children})"


class Hierarchy:
    def __init__(self):
        self.nodes = {}
        self.root = None

    def __len__(self):
        return len(self.nodes.keys())

    def add_node(self, node_id, name, parent=None):
        self.nodes[node_id] = Node(node_id, name, parent)
        self.nodes[node_id].depth = self.depth(node_id)
        if parent is not None:
            self.nodes[parent].add_child(node_id)
        else:
            self.root = node_id

    def children(self, node_id):
        return self.nodes[node_id].children

    def parent(self, node_id):
        return self.nodes[node_id].parent

    def is_leaf(self, node_id):
        return len(self.nodes[node_id].children) == 0

    def depth(self, node_id):
        d = 0
        cur = node_id
        while self.nodes[cur].parent is not None:
            d += 1
            cur = self.nodes[cur].parent
        return d

    def get_path_to_root(self, leaf_id):
        """
        Get the path from root to a leaf node.
        
        Args:
            leaf_id: The ID of the leaf node (or any node in the hierarchy)
        
        Returns:
            List of node IDs from root to leaf, e.g., [root_id, ..., leaf_id]
        """
        if leaf_id not in self.nodes:
            raise ValueError(f"Node {leaf_id} not found in hierarchy")
        
        path = []
        current_id = leaf_id
        
        # Traverse from leaf to root using parent pointers
        while current_id is not None:
            path.append(self.nodes[current_id].node_id)
            current_id = self.nodes[current_id].parent
        
        # Reverse to get path from root to leaf
        return list(reversed(path))

    def descendants(self, node_id):
        result = []

        def dfs(n):
            for c in self.nodes[n].children:
                result.append(c)
                dfs(c)

        dfs(node_id)
        return result

    def subtree_leaves(self, node_id):
        leaves = []

        def dfs(n):
            if self.is_leaf(n):
                leaves.append(n)
            for c in self.nodes[n].children:
                dfs(c)

        dfs(node_id)
        return leaves

    def get_leaf_index(self):
        """
        Returns a list where each position corresponds to a node (ordered by node_id),
        with 1 if the node is a leaf, 0 otherwise.
        
        e.g. if nodes 5 and 6 are leaves → [0, 0, 0, 0, 0, 1, 1]
        """
        sorted_nodes = sorted(self.nodes.values(), key=lambda n: n.node_id)
        return [1 if self.is_leaf(n.node_id) else 0 for n in sorted_nodes]

In [3]:
set_seed(SEED)
dataset = ImageDataset(
    data_directory = data_directory,
    data_subdirectories = data_subdirectories,
    class_names = ZOOPLANKTON_CLASSES,
    max_class_size = MAX_CLASS_SIZE,
    image_resolution = RESOLUTION,
    image_transforms = None,
    format_file = '.tif',
    seed = SEED
    )

hier_dataset = HierImageDataset(
    base_dataset=dataset,
    adjacency_graph = hier_adjacency_graph,
    levels=3,
    leaves_only=True
)
hier_dataset.print_dataset_details()

[leaves_only] Kept 59810 samples | Removed 1491 non-leaf samples

Total Dataset: Size = 59810 | Levels = 3
all_node_counts: {0: 59810, 1: 37350, 3: 27778, 10: 10000, 11: 10000, 12: 1728, 13: 6050, 4: 6522, 14: 3023, 15: 3499, 6: 3050, 2: 22460, 5: 10000, 7: 4858, 8: 1147, 9: 6455}

nodes_by_level: {0: [0], 1: [1, 2], 2: [3, 4, 6, 5, 7, 8, 9], 3: [10, 11, 12, 13, 14, 15]}


------------------------Level 0------------------------
Level: 0 | Class Name: root                 | Class Label:   0 | Type: Parent | Count:  59810 | Prop: 1.00

------------------------Level 1------------------------
Level: 1 | Class Name: Zoop-yes             | Class Label:   1 | Type: Parent | Count:  37350 | Prop: 0.62
Level: 1 | Class Name: Zoop-No              | Class Label:   2 | Type: Parent | Count:  22460 | Prop: 0.38

------------------------Level 2------------------------
Level: 2 | Class Name: Copepoda             | Class Label:   3 | Type: Parent | Count:  27778 | Prop: 0.46
Level: 2 | Class Name: Cla

In [4]:
from torch.utils.data import Dataset, Subset, DataLoader, SequentialSampler, WeightedRandomSampler, random_split
train_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(180),
    transforms.Pad(padding = 5, fill = 0),
    transforms.Resize((RESOLUTION, RESOLUTION)),
    transforms.ToTensor(),
])


hier_dataset.append_image_transforms(
    image_transforms = train_transforms, replace = True
)

TRAIN_PROP = 0.7
VAL_PROP = 0.1
TEST_PROP = 0.2

BATCH_SIZE = 64

train_split, val_split, test_split = hier_dataset.split_train_test_val(
    train_prop = TRAIN_PROP, val_prop = VAL_PROP, test_prop = TEST_PROP
)

# Create dataloaders
train_loader, val_loader, test_loader = hier_dataset.create_dataloaders(
    batch_size = BATCH_SIZE,
    train_indices = train_split,
    val_indices = val_split,
    test_indices = test_split,
    image_transforms = None,
    train_sample_weights = None
)


In [7]:
class Expert(nn.Module):
    def __init__(self, in_dim, num_children, mode="soft"):
        super().__init__()
        self.linear = nn.Linear(in_dim, num_children)
        self.mode = mode

    def forward(self, x):
        logits = self.linear(x)

        if self.mode == "soft":
            weights = torch.softmax(logits, dim=-1)
            return weights, logits

        elif self.mode == "hard":
            indices = torch.argmax(logits, dim=-1)
            return indices, logits

        else:
            raise ValueError("mode must be 'soft' or 'hard'")

class HierMoeNet(nn.Module):
    def __init__(self, hierarchy, label_to_id, weights_directory=None, seed=123):
        super().__init__()
        self.hierarchy = hierarchy
        self.seed = seed
        self.label_to_id = label_to_id
        backbone = models.efficientnet_b0(weights = weights_directory)
        self.shared = backbone.features
        self.pool = backbone.avgpool
        self.feature_dim = backbone.classifier[1].in_features
        self.local_classifiers = nn.ModuleDict()
        for node_id, node in hierarchy.nodes.items():
            if len(node.children) > 0:
                self.local_classifiers[str(node_id)] = Expert(self.feature_dim,len(node.children))
        self.leaf_index = torch.tensor(hierarchy.get_leaf_index(), dtype=torch.float32)
        self.bce = nn.BCELoss(reduction='none')


    def forward(self, x):
        # --- Backbone ---
        x = self.shared(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)                    # (B, feature_dim)

        # --- Run all local classifiers in one pass ---
        # Store conditional probs for every internal node
        # node_probs[node_id] -> (B, num_children) soft probabilities
        node_probs = {}
        node_logits = {}
        for node_id, classifier in self.local_classifiers.items():
            probs, logits = classifier.forward(x)   # (B, num_children)
            node_probs[node_id] = probs
            node_logits[node_id] = logits

        # --- Compute leaf probabilities via path products ---
        # For each leaf, walk root -> leaf and multiply conditional probs
        # P(leaf) = P(child_k | node_n) * P(node_n | node_n-1) * ... * P(node_1 | root)
        leaf_probs = [-1] * len(self.hierarchy.nodes)
        # print(leaf_probs)
        for node_id, node in self.hierarchy.nodes.items():
            # print(self.hierarchy.parent(node_id))
            # if self.hierarchy.parent(node_id) is None:
            #     continue
            # print(f"current node: {node.name}")
            path = self.hierarchy.get_path_to_root(node_id)
            # print(path)
            # path = [root, ..., parent_of_leaf, leaf]

            # Start with prob=1 and multiply down the path
            B = x.shape[0]
            prob = torch.ones(B, device=x.device)   # (B,)

            for i in range(len(path) - 1):
                parent = path[i]
                child  = path[i + 1]

                child_idx = self.hierarchy.nodes[parent].children.index(child)

                prob = prob * node_probs[str(parent)][:, child_idx]  # (B,)
            # print(prob)
            leaf_probs[node_id] = prob  # (B,)

        logit_tensor = torch.stack(leaf_probs).transpose(0,1)

        return logit_tensor, node_logits


    def loss_fn(self, logits, targets):
        leaf_index = self.leaf_index.to(logits.device)
        loss = self.bce(logits, targets) * leaf_index
        loss = loss.sum(dim=1) / self.leaf_index.sum()
        return loss.mean()



if torch.backends.mps.is_available():
    device = torch.device('mps')
    print(f'Using device: MPS (Apple Silicon GPU)')
elif torch.cuda.is_available():
    device = torch.device('cuda')
    print(f'Using device: CUDA GPU')
else:
    device = torch.device('cpu')
    print(f'Using device: CPU')

HierMoeNet = HierMoeNet(hier_dataset.hierarchy, hier_dataset.label_to_ids)
# result = HierMoeNet.forward(n['image'])

Using device: MPS (Apple Silicon GPU)


In [8]:
from sklearn.metrics import f1_score
class Trainer:
    def __init__(self, learning_rate, max_epochs, gradient_clip_val=1, device="cpu", print_every: int=5):
        self.learning_rate = learning_rate
        self.max_epochs = max_epochs
        self.gradient_clip_val = gradient_clip_val
        self.device = device
        self.train_loss = []  
        self.valid_loss = []  
        self.train_acc = []
        self.valid_acc = []
        self.train_f1 = []
        self.valid_f1 = []
        self.print_every = print_every

    @staticmethod
    def clip_gradients(model, max_norm):
        if max_norm:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm)
    
    @staticmethod
    def compute_metrics(leaf_probs, label_node, leaf_index):
        """
        Converts soft predictions and targets to hard class predictions
        over leaf nodes only, then computes accuracy and macro F1.

        Args:
            leaf_probs  : (B, N) predicted probabilities
            label_node  : (B, N) soft ground truth labels
            leaf_index  : (N,)   binary mask — 1 for leaf nodes

        Returns:
            accuracy (float), f1 (float)
        """
        leaf_mask = leaf_index.bool()                               # (N,)

        # Restrict to leaf columns only
        leaf_probs_only  = leaf_probs[:, leaf_mask]                 # (B, N_leaves)
        label_node_only  = label_node[:, leaf_mask]                 # (B, N_leaves)

        pred_classes  = leaf_probs_only.argmax(dim=1).cpu()         # (B,)
        target_classes = label_node_only.argmax(dim=1).cpu()        # (B,)

        accuracy = (pred_classes == target_classes).float().mean().item()
        f1 = f1_score(target_classes.numpy(), pred_classes.numpy(), average="macro", zero_division=0)

        return accuracy, f1


    def fit(self, model, train_loader, valid_loader, optimizer=None):
        print(f"Training at device: {self.device}")
        model.to(self.device)
        if optimizer is None:
            optimizer = torch.optim.Adam(model.parameters(), lr=self.learning_rate)
        print("beginning training")
        for epoch in range(self.max_epochs):
            # print(f"Epoch: {epoch + 1}")
            model.train()
            train_loss = 0
            train_preds, train_targets = [], []
            for i, batch in enumerate(train_loader):
                # if i % 10 == 0:
                #     print(i)
                optimizer.zero_grad()
                image, label_node, path, targets, masks = batch.values()
                image, label_node = image.to(self.device), label_node.to(self.device)
                # print("collected image")
                leaf_tensor, node_logits = model(image)
                # print("Output model")
                
                loss = model.loss_fn(leaf_tensor, label_node)
                # print("loss calculated")
                loss.backward()
                self.clip_gradients(model, self.gradient_clip_val)
                optimizer.step()
                # print("optimizer step")
                train_loss += loss.detach().cpu().item()

                leaf_mask = model.leaf_index.bool()
                train_preds.append(leaf_tensor[:, leaf_mask].argmax(dim=1).detach().cpu())
                train_targets.append(label_node[:, leaf_mask].argmax(dim=1).detach().cpu())

            avg_train_loss = train_loss / len(train_loader)
            train_preds   = torch.cat(train_preds).numpy()
            train_targets = torch.cat(train_targets).numpy()
            train_acc = (train_preds == train_targets).mean()
            train_f1  = f1_score(train_targets, train_preds, average="macro", zero_division=0)

            self.train_loss.append(avg_train_loss)
            self.train_acc.append(train_acc)
            self.train_f1.append(train_f1)


            # --- Validation ---
            model.eval()
            valid_loss = 0
            valid_preds, valid_targets = [], []

            with torch.no_grad():
                for batch in valid_loader:
                    image = batch["image"].to(self.device)
                    label_node = batch["label_node"].to(self.device)

                    leaf_tensor, node_logits = model(image)
                    loss = model.loss_fn(leaf_tensor, label_node)
                    valid_loss += loss.detach().cpu().item()

                    leaf_mask = model.leaf_index.bool()
                    valid_preds.append(leaf_tensor[:, leaf_mask].argmax(dim=1).cpu())
                    valid_targets.append(label_node[:, leaf_mask].argmax(dim=1).cpu())

            avg_valid_loss = valid_loss / len(valid_loader)
            valid_preds   = torch.cat(valid_preds).numpy()
            valid_targets = torch.cat(valid_targets).numpy()
            valid_acc = (valid_preds == valid_targets).mean()
            valid_f1  = f1_score(valid_targets, valid_preds, average="macro", zero_division=0)

            self.valid_loss.append(avg_valid_loss)
            self.valid_acc.append(valid_acc)
            self.valid_f1.append(valid_f1)

            # --- Logging ---
            if (epoch % self.print_every == 0) or (epoch == self.max_epochs):
                print(
                    f"Epoch [{epoch+1}/{self.max_epochs}] "
                    f"| Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.4f} | Train F1: {train_f1:.4f} "
                    f"| Valid Loss: {avg_valid_loss:.4f} | Valid Acc: {valid_acc:.4f} | Valid F1: {valid_f1:.4f}"
                )

    def predict(self, model, pred_loader):
        model.to(self.device)
        model.eval()
        leaf_index = model.leaf_index.to(self.device)
        leaf_mask  = leaf_index.bool()

        all_leaf_probs  = []
        all_label_nodes = []
        all_preds       = []
        all_targets     = []

        with torch.no_grad():
            for batch in pred_loader:
                image      = batch["image"].to(self.device)
                label_node = batch["label_node"].to(self.device)

                leaf_tensor, _ = model(image)                                       # (B, N)

                all_leaf_probs.append(leaf_tensor.cpu())
                all_label_nodes.append(label_node.cpu())
                all_preds.append(leaf_tensor[:, leaf_mask].argmax(dim=1).cpu())
                all_targets.append(label_node[:, leaf_mask].argmax(dim=1).cpu())

        all_preds   = torch.cat(all_preds).numpy()
        all_targets = torch.cat(all_targets).numpy()

        accuracy = (all_preds == all_targets).mean()
        f1       = f1_score(all_targets, all_preds, average="macro", zero_division=0)

        print(f"Predict | Accuracy: {accuracy:.4f} | F1: {f1:.4f}")

        return {
            "leaf_probs"  : torch.cat(all_leaf_probs, dim=0),
            "label_node"  : torch.cat(all_label_nodes, dim=0),
            "predictions" : all_preds,
            "targets"     : all_targets,
            "accuracy"    : accuracy,
            "f1"          : f1,
        }

In [9]:
# train_acc = 0.713
# valid_acc = 0.729
# 7 epochs took 16 minutes
HYPERPARAMETERS = {
    # 'optimizer': 'Adam', 
    'lr': 3e-4, 
    'epochs': 1, 
    'scheduler':{'type': 'CosineAnnealingLR', 'T_max': 50},
    'early_stopping': {'patience': 15, 'delta': 0.0005},
}

trainer = Trainer(learning_rate=HYPERPARAMETERS['lr'], max_epochs=HYPERPARAMETERS['epochs'], device=device,
                  print_every=1)
trainer.fit(HierMoeNet, train_loader, val_loader)

Training at device: mps
beginning training
Epoch [1/1] | Train Loss: 0.1763 | Train Acc: 0.5057 | Train F1: 0.4036 | Valid Loss: 0.1396 | Valid Acc: 0.6267 | Valid F1: 0.5090


In [11]:
trainer.valid_acc

[np.float64(0.8162514629660592),
 np.float64(0.8221033272028089),
 np.float64(0.8222705233238589),
 np.float64(0.8184250125397091),
 np.float64(0.8242768767764588),
 np.float64(0.8312991138605584),
 np.float64(0.8266176224711587),
 np.float64(0.819260993144959),
 np.float64(0.8328038789500084),
 np.float64(0.819260993144959),
 np.float64(0.8465139608761076),
 np.float64(0.8473499414813577),
 np.float64(0.8294599565290085),
 np.float64(0.840327704397258),
 np.float64(0.8451763919077078)]

In [12]:
# .77 acc, 0.72 f1
# Predict | Accuracy: 0.8200 | F1: 0.8120
trainer.predict(HierMoeNet, test_loader)

Predict | Accuracy: 0.8552 | F1: 0.8537


{'leaf_probs': tensor([[1.0000e+00, 9.9781e-01, 2.1888e-03,  ..., 9.9780e-01, 1.0292e-05,
          2.5476e-12],
         [1.0000e+00, 4.9471e-05, 9.9995e-01,  ..., 4.7737e-05, 3.6513e-07,
          1.4918e-08],
         [1.0000e+00, 2.7649e-04, 9.9972e-01,  ..., 2.5167e-04, 6.7004e-06,
          1.2380e-06],
         ...,
         [1.0000e+00, 9.9745e-01, 2.5455e-03,  ..., 1.4615e-04, 1.8530e-05,
          6.2967e-04],
         [1.0000e+00, 2.8255e-04, 9.9972e-01,  ..., 2.7569e-04, 2.5532e-07,
          3.7580e-09],
         [1.0000e+00, 9.9649e-01, 3.5129e-03,  ..., 7.6994e-05, 1.9828e-04,
          2.5802e-03]]),
 'label_node': tensor([[0., 0., 0.,  ..., 1., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         ...,
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]]),
 'predictions': array([8, 4, 0, ..., 6, 0, 5]),
 'targets': array([8, 4, 0, ..., 6, 0, 5]),
 'accura

# Archive

In [31]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
from tqdm import tqdm
import os
from collections import defaultdict

class HardMoeTrainer:
    """
    Trainer class for HardMoeNet model.
    
    Args:
        model: HardMoeNet model instance
        train_loader: DataLoader for training data
        val_loader: DataLoader for validation data
        hyperparameters: Dictionary containing training hyperparameters
        device: Device to train on ('cuda', 'mps', or 'cpu')
        checkpoint_dir: Directory to save model checkpoints
    """
    
    def __init__(self, model, train_loader, val_loader, hyperparameters, device, checkpoint_dir='checkpoints'):
        self.model = model.to(device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.device = device
        self.checkpoint_dir = checkpoint_dir
        
        # Create checkpoint directory
        os.makedirs(checkpoint_dir, exist_ok=True)
        
        # Hyperparameters
        self.epochs = hyperparameters['epochs']
        self.lr = hyperparameters['lr']
        
        # Optimizer
        if hyperparameters['optimizer'] == 'Adam':
            self.optimizer = optim.Adam(model.parameters(), lr=self.lr)
        elif hyperparameters['optimizer'] == 'SGD':
            self.optimizer = optim.SGD(model.parameters(), lr=self.lr, momentum=0.9)
        else:
            raise ValueError(f"Unsupported optimizer: {hyperparameters['optimizer']}")
        
        # Scheduler
        if hyperparameters.get('scheduler') is not None:
            scheduler_config = hyperparameters['scheduler']
            if scheduler_config['type'] == 'CosineAnnealingLR':
                self.scheduler = optim.lr_scheduler.CosineAnnealingLR(
                    self.optimizer, T_max=scheduler_config['T_max']
                )
            elif scheduler_config['type'] == 'ReduceLROnPlateau':
                self.scheduler = optim.lr_scheduler.ReduceLROnPlateau(
                    self.optimizer, mode='max', factor=0.5, patience=5
                )
            else:
                self.scheduler = None
        else:
            self.scheduler = None
        
        # Early stopping
        if hyperparameters.get('early_stopping') is not None:
            self.early_stopping_patience = hyperparameters['early_stopping']['patience']
            self.early_stopping_delta = hyperparameters['early_stopping']['delta']
            self.use_early_stopping = True
        else:
            self.use_early_stopping = False
        
        # Tracking
        self.train_losses = []
        self.val_losses = []
        self.val_accuracies = []
        self.best_val_accuracy = 0.0
        self.best_epoch = 0
        self.epochs_without_improvement = 0
        
    def train_epoch(self):
        """Train for one epoch"""
        self.model.train()
        
        epoch_loss = 0.0
        epoch_node_losses = defaultdict(float)
        num_batches = 0
        
        progress_bar = tqdm(self.train_loader, desc='Training')
        
        for batch_idx, batch in enumerate(progress_bar):
            # Move batch to device
            images, labels = batch
            images = images.to(self.device)
            labels = labels.to(self.device)
            batch = (images, labels)
            
            # Zero gradients
            self.optimizer.zero_grad()
            
            # Forward pass
            result = self.model(batch, stage='train')
            loss = result['loss']
            print(f"loss: {loss}")
            # Backward pass
            loss.backward()
            self.optimizer.step()
            
            # Track losses
            epoch_loss += loss.item()
            for node_key, node_loss in result['node_losses'].items():
                epoch_node_losses[node_key] += node_loss.item()
            num_batches += 1
            
            # Update progress bar
            progress_bar.set_postfix({'loss': loss.item()})
        
        # Average losses
        avg_loss = epoch_loss / num_batches
        avg_node_losses = {k: v / num_batches for k, v in epoch_node_losses.items()}
        
        return avg_loss, avg_node_losses
    
    def validate(self):
        """Validate the model"""
        self.model.eval()
        
        val_loss = 0.0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for batch in tqdm(self.val_loader, desc='Validating'):
                images, labels = batch
                images = images.to(self.device)
                labels = labels.to(self.device)
                
                # Get predictions (inference mode)
                features = self.model.shared(images)
                features = self.model.pool(features)
                features = torch.flatten(features, 1)
                batch_size = features.size(0)
                
                predictions = self._inference(features, batch_size)
                
                # Calculate accuracy
                correct += (predictions == labels).sum().item()
                total += labels.size(0)
                
                # Calculate loss using training forward pass
                batch_device = (images, labels)
                result = self.model(batch_device, stage='val')
                val_loss += result['loss'].item()
        
        avg_val_loss = val_loss / len(self.val_loader)
        accuracy = correct / total
        
        return avg_val_loss, accuracy
    
    def _inference(self, features, batch_size):
        """
        Inference mode: follow predicted path from root to leaf
        """
        current_nodes = torch.full(
            (batch_size,), 
            self.model.hierarchy.root_id, 
            dtype=torch.long, 
            device=features.device
        )
        active_mask = torch.ones(batch_size, dtype=torch.bool, device=features.device)
        
        max_depth = self.model.hierarchy.max_depth
        
        for depth in range(max_depth + 1):
            if not active_mask.any():
                break
            
            unique_nodes = current_nodes[active_mask].unique()
            
            for node_id in unique_nodes:
                node_id_val = node_id.item()
                node = self.model.hierarchy.nodes[node_id_val]
                
                # Skip leaf nodes
                if len(node.children) == 0:
                    active_mask[current_nodes == node_id] = False
                    continue
                
                # Find samples at this node
                node_mask = (current_nodes == node_id) & active_mask
                node_indices = torch.where(node_mask)[0]
                
                if len(node_indices) == 0:
                    continue
                
                # Batch forward pass
                node_key = str(node_id_val)
                batch_features = features[node_indices]
                logits = self.model.local_classifiers[node_key](batch_features)
                
                # Select child with highest logit
                child_indices = torch.argmax(logits, dim=1)
                
                # Update current nodes
                for i, idx in enumerate(node_indices):
                    child_node_id = node.children[child_indices[i].item()]
                    current_nodes[idx] = child_node_id
        
        return current_nodes
    
    def train(self):
        """Main training loop"""
        print(f"\n{'='*60}")
        print(f"Starting Training")
        print(f"{'='*60}")
        print(f"Device: {self.device}")
        print(f"Epochs: {self.epochs}")
        print(f"Learning Rate: {self.lr}")
        print(f"Train Batches: {len(self.train_loader)}")
        print(f"Val Batches: {len(self.val_loader)}")
        print(f"{'='*60}\n")
        
        for epoch in range(self.epochs):
            print(f"\nEpoch {epoch + 1}/{self.epochs}")
            print("-" * 60)
            
            # Training
            train_loss, train_node_losses = self.train_epoch()
            self.train_losses.append(train_loss)
            
            # Validation
            val_loss, val_accuracy = self.validate()
            self.val_losses.append(val_loss)
            self.val_accuracies.append(val_accuracy)
            
            # Update learning rate scheduler
            if self.scheduler is not None:
                if isinstance(self.scheduler, optim.lr_scheduler.ReduceLROnPlateau):
                    self.scheduler.step(val_accuracy)
                else:
                    self.scheduler.step()
            
            # Print epoch summary
            print(f"\nEpoch {epoch + 1} Summary:")
            print(f"  Train Loss: {train_loss:.4f}")
            print(f"  Val Loss: {val_loss:.4f}")
            print(f"  Val Accuracy: {val_accuracy:.4f}")
            
            # Print node-specific losses
            print(f"  Node Losses:")
            for node_key, node_loss in sorted(train_node_losses.items()):
                print(f"    Node {node_key}: {node_loss:.4f}")
            
            if self.scheduler is not None:
                current_lr = self.optimizer.param_groups[0]['lr']
                print(f"  Learning Rate: {current_lr:.6f}")
            
            # Save best model
            if val_accuracy > self.best_val_accuracy + (self.early_stopping_delta if self.use_early_stopping else 0):
                self.best_val_accuracy = val_accuracy
                self.best_epoch = epoch + 1
                self.epochs_without_improvement = 0
                
                checkpoint_path = os.path.join(
                    self.checkpoint_dir, 
                    f'best_model_acc_{val_accuracy:.4f}.pth'
                )
                torch.save({
                    'epoch': epoch + 1,
                    'model_state_dict': self.model.state_dict(),
                    'optimizer_state_dict': self.optimizer.state_dict(),
                    'val_accuracy': val_accuracy,
                    'val_loss': val_loss,
                    'train_loss': train_loss,
                }, checkpoint_path)
                
                print(f"  ✓ New best model saved! (Accuracy: {val_accuracy:.4f})")
            else:
                self.epochs_without_improvement += 1
                print(f"  No improvement for {self.epochs_without_improvement} epoch(s)")
            
            # Early stopping
            if self.use_early_stopping and self.epochs_without_improvement >= self.early_stopping_patience:
                print(f"\n{'='*60}")
                print(f"Early stopping triggered after {epoch + 1} epochs")
                print(f"Best validation accuracy: {self.best_val_accuracy:.4f} at epoch {self.best_epoch}")
                print(f"{'='*60}\n")
                break
        
        print(f"\n{'='*60}")
        print(f"Training Complete!")
        print(f"Best Validation Accuracy: {self.best_val_accuracy:.4f} (Epoch {self.best_epoch})")
        print(f"{'='*60}\n")
        
        return {
            'train_losses': self.train_losses,
            'val_losses': self.val_losses,
            'val_accuracies': self.val_accuracies,
            'best_val_accuracy': self.best_val_accuracy,
            'best_epoch': self.best_epoch
        }
    
    def load_best_model(self):
        """Load the best saved model"""
        checkpoint_files = [f for f in os.listdir(self.checkpoint_dir) if f.startswith('best_model')]
        if not checkpoint_files:
            print("No checkpoint found!")
            return None
        
        # Get the latest checkpoint
        latest_checkpoint = max(
            checkpoint_files, 
            key=lambda f: os.path.getctime(os.path.join(self.checkpoint_dir, f))
        )
        checkpoint_path = os.path.join(self.checkpoint_dir, latest_checkpoint)
        
        checkpoint = torch.load(checkpoint_path, map_location=self.device)
        self.model.load_state_dict(checkpoint['model_state_dict'])
        
        print(f"Loaded best model from {checkpoint_path}")
        print(f"  Epoch: {checkpoint['epoch']}")
        print(f"  Val Accuracy: {checkpoint['val_accuracy']:.4f}")
        print(f"  Val Loss: {checkpoint['val_loss']:.4f}")
        
        return checkpoint


HardNet = HardMoeNet(hier_dataset.hierarchy)

# Create trainer
trainer = HardMoeTrainer(
    model=HardNet,
    train_loader=train_loader,
    val_loader=val_loader,
    hyperparameters=HYPERPARAMETERS,
    device=device,
    checkpoint_dir='checkpoints'
)

# Train
history = trainer.train()



Starting Training
Device: mps
Epochs: 10
Learning Rate: 0.0005
Train Batches: 671
Val Batches: 96


Epoch 1/10
------------------------------------------------------------


Training:   0%|          | 1/671 [00:00<10:24,  1.07it/s, loss=0]

loss: 0.0


Training:   0%|          | 3/671 [00:01<03:41,  3.02it/s, loss=0]

loss: 0.0
loss: 0.0


...

loss: 0.0
loss: 0.0
loss: 0.0


Training:  15%|█▌        | 102/671 [00:12<00:55, 10.19it/s, loss=0]

loss: 0.0


Training:  15%|█▌        | 102/671 [00:13<01:13,  7.78it/s, loss=0]


KeyboardInterrupt: 

In [11]:
HardNet.local_classifiers

ModuleDict(
  (root): Linear(in_features=1280, out_features=2, bias=True)
  (Zoop-yes): Linear(in_features=1280, out_features=3, bias=True)
  (Zoop-No): Linear(in_features=1280, out_features=3, bias=True)
  (Copepoda): Linear(in_features=1280, out_features=4, bias=True)
  (Cladocera): Linear(in_features=1280, out_features=2, bias=True)
)

In [ ]:
if torch.backends.mps.is_available():
    device = torch.device('mps')

HardNet.to(device)

In [7]:
model = models.efficientnet_b0(weights = None)
model.classifier[1]

Linear(in_features=1280, out_features=1000, bias=True)

In [8]:
model.classifier

Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=1000, bias=True)
)

In [13]:
model.features[1]

Sequential(
  (0): MBConv(
    (block): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): SiLU(inplace=True)
      )
      (1): SqueezeExcitation(
        (avgpool): AdaptiveAvgPool2d(output_size=1)
        (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
        (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
        (activation): SiLU(inplace=True)
        (scale_activation): Sigmoid()
      )
      (2): Conv2dNormActivation(
        (0): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (stochastic_depth): StochasticDepth(p=0.0, mode=row)
  )
)